# 06 — Direct Preference Optimization (DPO) Training & Evaluation (1.3B)
**Goal**: Train `deepseek-ai/deepseek-coder-1.3b-instruct` on execution-grounded `(chosen, rejected)` pairs from the SFT model (`β=0.1`, LR `5e-5`, 3 epochs) using the unified `normalize_tests` harness. Produces `./checkpoints/dpo/final` and runs the DPO evaluation benchmark on HumanEval and MBPP ($K=1, 3, 5$).

**Paper run**: use `train[:500]` (or larger) for preference collection — `train[:50]` is smoke only.

---


## Step 1: Environment Setup & Universal Path Resolution

In [ ]:
!pip install -q "trl<0.12.0" peft transformers datasets bitsandbytes accelerate pandas

import sys, os, shutil

repo_root = os.path.abspath(os.getcwd())
GIT_SHA = ''
if not os.path.exists(os.path.join(repo_root, 'src', 'models', 'loader.py')):
    if os.path.exists(os.path.join(os.path.abspath('..'), 'src', 'models', 'loader.py')):
        repo_root = os.path.abspath('..')
    elif os.path.isdir('/kaggle/working'):
        !rm -rf /kaggle/working/src
        !git clone -b Rajdeep https://github.com/Oin19/self-correction-slm.git temp_repo || git clone https://github.com/Oin19/self-correction-slm.git temp_repo || git clone -b junior-A https://github.com/Oin19/self-correction-llm-rl.git temp_repo
        !cp -r temp_repo/src /kaggle/working/src
        _sha = !git -C temp_repo rev-parse HEAD
        GIT_SHA = _sha[0] if _sha else ''
        print(f'Cloned repo @ {GIT_SHA}')
        !rm -rf temp_repo
        repo_root = '/kaggle/working'

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print(f"Project path initialized: {repo_root}")

!pip uninstall -y torchao
try:
    import peft.import_utils
    peft.import_utils.is_torchao_available = lambda: False
except Exception:
    pass
try:
    import peft.tuners.lora.torchao
    peft.tuners.lora.torchao.is_torchao_available = lambda: False
except Exception:
    pass

import torch
import pandas as pd
from datasets import load_dataset
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from src.models.loader import load_model_and_tokenizer
from src.training.dpo import make_preference_pairs, run_dpo_training, parse_apps_test_cases
from src.utils.test_cases import normalize_tests
from src.evaluation.metrics import evaluate

os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("Environment initialized! (normalize_tests harness active)")


## Step 2: Collect Preference Trajectory Pairs `(prompt, chosen, rejected)`

Loads the trained **SFT parent adapter** (`./checkpoints/sft/final`) via `PeftModel.from_pretrained` (preserving trained SFT weights) and runs $K=3$ debug rollouts with per-problem JSONL resume checkpointing.

In [ ]:
MODEL_NAME = "deepseek-ai/deepseek-coder-1.3b-instruct"
SFT_CHECKPOINT = "./checkpoints/sft/final"
PREF_SPLIT = "train[:500]"   # paper: 500+; smoke: train[:50]
SEED = 42
K_TURNS = 3
PAIR_CKPT = f"./dpo_pairs_{PREF_SPLIT.replace('[:', '_').replace(']', '')}_seed{SEED}.jsonl"

search_paths = [
    "./checkpoints/sft/final",
    "/kaggle/working/checkpoints/sft/final",
    "/content/checkpoints/sft/final",
    "/kaggle/input",
]

model_path = None
fallback_adapter = None
for p in search_paths:
    if os.path.exists(p):
        if os.path.exists(os.path.join(p, "adapter_config.json")):
            model_path = p
            break
        for root, dirs, files in os.walk(p):
            if "adapter_config.json" in files:
                if "sft" in root.lower():
                    model_path = root
                    break
                elif fallback_adapter is None and "ppo" not in root.lower() and "dpo" not in root.lower():
                    fallback_adapter = root
    if model_path:
        break

if not model_path and fallback_adapter:
    model_path = fallback_adapter

if not model_path:
    print("No pre-saved SFT adapter found in /kaggle/input. Training SFT parent checkpoint now...")
    from src.training.sft import run_sft_training
    sft_apps = load_dataset("codeparrot/apps", revision="refs/convert/parquet", split="train[:2000]")
    sft_clean = sft_apps.filter(lambda x: len(x["solutions"]) > 0)
    sft_model, sft_tok = load_model_and_tokenizer(model_name=MODEL_NAME, load_in_4bit=False, lora_r=16)
    run_sft_training(
        model=sft_model,
        tokenizer=sft_tok,
        dataset=sft_clean,
        output_dir="./checkpoints/sft",
        num_epochs=1,
        per_device_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-5,
    )
    del sft_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    model_path = "./checkpoints/sft/final"

print(f"Loading SFT parent adapter from {model_path} for DPO pair collection in FP16 precision...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)
model = PeftModel.from_pretrained(base_model, model_path, is_trainable=True)
print("Successfully restored trained SFT adapter for DPO!")

print(f"\nLoading APPS {PREF_SPLIT} for preference pair generation...")
apps = load_dataset("codeparrot/apps", revision="refs/convert/parquet", split=PREF_SPLIT)
apps_clean = apps.filter(lambda x: len(normalize_tests(x)) > 0, load_from_cache_file=False)
print(f"Prepared {len(apps_clean)} problems with executable tests.")

print(f"Generating preference dataset... (checkpoint: {PAIR_CKPT})")
preference_data = make_preference_pairs(
    apps_clean, model, tokenizer, K=K_TURNS, seed=SEED,
    checkpoint_path=PAIR_CKPT, max_gen_time=90,
)
print(f"Total DPO preference pairs generated: {len(preference_data)}")
if len(preference_data) == 0:
    raise ValueError("No execution-grounded DPO preference pairs were generated; refusing to train on an empty dataset.")


## Step 3: Run DPO Training

Includes compatibility shims for `trl<0.12` + Kaggle's newer `transformers` (`MODEL_FOR_VISION_2_SEQ_MAPPING_NAMES`, `processing_class`, `Trainer.get_batch_samples`, and `num_items_in_batch`) so training runs cleanly without restarting your session.

In [ ]:
import importlib, inspect
import transformers
import transformers.trainer as _hf_trainer_mod
import transformers.models.auto.modeling_auto as _modeling_auto

# 1. Shim removed vision mapping constant in newer transformers for trl<0.12
if not hasattr(_modeling_auto, "MODEL_FOR_VISION_2_SEQ_MAPPING_NAMES"):
    _modeling_auto.MODEL_FOR_VISION_2_SEQ_MAPPING_NAMES = getattr(
        _modeling_auto, "MODEL_FOR_IMAGE_TEXT_TO_TEXT_MAPPING_NAMES", {}
    )

# 2. Reload Trainer and DPOTrainer cleanly to reset any prior cell monkey-patches
importlib.reload(_hf_trainer_mod)
transformers.Trainer = _hf_trainer_mod.Trainer
from transformers import Trainer

import trl
import trl.trainer.dpo_trainer as _trl_dpo_mod
importlib.reload(_trl_dpo_mod)
trl.DPOTrainer = _trl_dpo_mod.DPOTrainer
from trl import DPOConfig, DPOTrainer

# 3. Shim Trainer.__init__ using default-arg binding (_orig) so re-runs never recurse
if not getattr(Trainer.__init__, "_trl_compat_patched", False):
    _true_init = Trainer.__init__
    _sig = inspect.signature(_true_init)
    if "tokenizer" not in _sig.parameters and "processing_class" in _sig.parameters:
        def _patched_trainer_init(self, *args, _orig=_true_init, **kwargs):
            if "tokenizer" in kwargs and "processing_class" not in kwargs:
                kwargs["processing_class"] = kwargs.pop("tokenizer")
            elif "tokenizer" in kwargs:
                kwargs.pop("tokenizer")
            return _orig(self, *args, **kwargs)
        _patched_trainer_init._trl_compat_patched = True
        Trainer.__init__ = _patched_trainer_init

# 4. In trl<0.12, DPOTrainer.get_batch_samples(model, batch) was an eval helper that
#    shadowed transformers>=4.46 Trainer.get_batch_samples(epoch_iterator, num_batches, device).
if hasattr(Trainer, "get_batch_samples"):
    DPOTrainer.get_batch_samples = Trainer.get_batch_samples

# 5. Shim DPOTrainer.compute_loss for num_items_in_batch kwarg in newer transformers
if not getattr(DPOTrainer.compute_loss, "_trl_compat_patched", False):
    _true_cl = DPOTrainer.compute_loss
    _csig = inspect.signature(_true_cl)
    if "num_items_in_batch" not in _csig.parameters and not any(
        p.kind == inspect.Parameter.VAR_KEYWORD for p in _csig.parameters.values()
    ):
        def _patched_compute_loss(self, model, inputs, return_outputs=False, *args, _orig=_true_cl, **kwargs):
            return _orig(self, model, inputs, return_outputs=return_outputs)
        _patched_compute_loss._trl_compat_patched = True
        DPOTrainer.compute_loss = _patched_compute_loss

if not hasattr(model, "warnings_issued"):
    model.warnings_issued = {}

print("Starting DPO Training...")
dpo_trainer = run_dpo_training(
    model=model,
    tokenizer=tokenizer,
    preference_data=preference_data,
    output_dir="./checkpoints/dpo",
    beta=0.1,
    learning_rate=5e-5,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    seed=SEED,
    commit_sha=GIT_SHA if 'GIT_SHA' in dir() else '',
    preference_split=PREF_SPLIT,
    debug_turns_k=K_TURNS,
)

print("\nDPO Training completed successfully!")
print("Saved final DPO adapter checkpoint to ./checkpoints/dpo/final")


## Step 4: Checkpoint Verification & DPO Benchmark Evaluation (HumanEval & MBPP)
Verifies saved DPO adapter files, reloads weights cleanly via `PeftModel.from_pretrained`, runs APPS sanity checks, and evaluates the trained 1.3B DPO model across $K=1, 3, 5$ on HumanEval and MBPP.

In [ ]:
import os, json
import torch
import pandas as pd
from peft import PeftModel, PeftConfig
from transformers import AutoModelForCausalLM, AutoTokenizer

checkpoint_dir = None
search_roots = [
    './checkpoints/dpo/final',
    '/content/checkpoints/dpo/final',
    '/content/self-correction-slm/checkpoints/dpo/final',
    '/kaggle/working/checkpoints/dpo/final',
    '/kaggle/working',
    '/kaggle/input'
]

for root in search_roots:
    if os.path.exists(root):
        if os.path.exists(os.path.join(root, 'adapter_config.json')):
            checkpoint_dir = root
            break
        for r, dirs, files in os.walk(root):
            if 'adapter_config.json' in files and 'dpo' in r.lower():
                checkpoint_dir = r
                break
    if checkpoint_dir:
        break

if not checkpoint_dir or not os.path.exists(os.path.join(checkpoint_dir, 'adapter_config.json')):
    raise FileNotFoundError("DPO checkpoint ./checkpoints/dpo/final not found!")

print(f"=== 1. Inspecting DPO Checkpoint Files in {checkpoint_dir} ===")
for fname in sorted(os.listdir(checkpoint_dir)):
    fpath = os.path.join(checkpoint_dir, fname)
    if os.path.isfile(fpath):
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        print(f"  - {fname}: {size_mb:.2f} MB")

print("\n=== 2. Verifying DPO LoRA Configuration ===")
config = PeftConfig.from_pretrained(checkpoint_dir)
print(f"  - lora_alpha: {getattr(config, 'lora_alpha', 'N/A')}")
print(f"  - r: {getattr(config, 'r', 'N/A')}")
print(f"  - target_modules: {list(getattr(config, 'target_modules', []))}")

print("\n=== 3. Reloading Saved DPO Model & Adapter ===")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)
reloaded_dpo = PeftModel.from_pretrained(base_model, checkpoint_dir, is_trainable=False)
reloaded_dpo.eval()
print("Reloaded DPO adapter model successfully!")

print("\n=== 4. Evaluating DPO Model on HumanEval & MBPP (K=1, 3, 5) ===")
os.makedirs("./results", exist_ok=True)
humaneval = load_dataset('openai_humaneval', split='test')
mbpp = load_dataset('mbpp', split='test')

EVAL_SLICE = 20  # Set to None for full benchmark evaluation
he_ds = humaneval.select(range(min(EVAL_SLICE, len(humaneval)))) if EVAL_SLICE else humaneval
mbpp_ds = mbpp.select(range(min(EVAL_SLICE, len(mbpp)))) if EVAL_SLICE else mbpp

he_dpo = evaluate(reloaded_dpo, tokenizer, he_ds, K=5, label="DPO HumanEval", output_jsonl="./results/dpo_humaneval_k5.jsonl")
mbpp_dpo = evaluate(reloaded_dpo, tokenizer, mbpp_ds, K=5, label="DPO MBPP", output_jsonl="./results/dpo_mbpp_k5.jsonl")

dpo_rows = []
for k in [1, 3, 5]:
    dpo_rows.append({
        "Model": "DPO Model (1.3B)",
        "K": k,
        "HumanEval Pass@1": f"{he_dpo['pass_at_1']:.2%}",
        "HumanEval Fix@K": f"{he_dpo.get(f'fix_at_{k}', 0.0):.2%}",
        "HumanEval CondRepair@K": f"{he_dpo.get(f'conditional_repair_at_{k}', 0.0):.2%}",
        "MBPP Pass@1": f"{mbpp_dpo['pass_at_1']:.2%}",
        "MBPP Fix@K": f"{mbpp_dpo.get(f'fix_at_{k}', 0.0):.2%}",
        "MBPP CondRepair@K": f"{mbpp_dpo.get(f'conditional_repair_at_{k}', 0.0):.2%}",
    })
df_dpo = pd.DataFrame(dpo_rows)
df_dpo.to_csv("./results/dpo_eval_summary.csv", index=False)
print("\n================ DPO 1.3B Evaluation Summary ================")
print(df_dpo.to_string(index=False))
print("Saved DPO evaluation results to ./results/dpo_eval_summary.csv")
